# Superconductivity: predicting $T_c$ from composition

In the last notebook we built logistic regression from scratch on synthetic two-dimensional
data, where we could *see* the decision boundary. Now we turn it on a real problem with 81
features, where we cannot see anything — and where the answer genuinely matters.

## What superconductivity is

Below a critical temperature **$T_c$**, certain materials carry direct current with **exactly
zero resistance** and expel magnetic flux from their interior (the Meissner effect). It is a
thermodynamic phase transition, not a gradual improvement: above $T_c$ the material is an
ordinary conductor, below it the resistance is not small but *zero*.

$T_c$ is the number that decides whether a material is a laboratory curiosity or usable
technology. Every application — MRI magnets, maglev, lossless transmission, superconducting
detectors — is gated by how much cryogenic plumbing it demands.

## Why predicting $T_c$ is an open problem

**BCS theory (1957)** explains conventional superconductivity: below $T_c$, electrons bind into
Cooper pairs through their interaction with lattice vibrations (phonons). It is a triumph —
and because the pairing is phonon-mediated, it also implies a rough ceiling, usually quoted
around **30–40 K** at ambient pressure.

That ceiling broke in **1986**, when Bednorz and Müller found superconductivity in a
copper-oxide ceramic. The **cuprates** climbed rapidly past the boiling point of liquid
nitrogen (77 K) — the practical watershed, since nitrogen is cheap and helium is not. The
best sit near 135 K. A second unconventional family, the **iron-based** superconductors,
turned up in 2008.

**Neither family has a settled microscopic theory.** Forty years on, there is still no way to
compute $T_c$ from first principles for a cuprate. So the field does something else: measure
thousands of materials, describe each by its composition, and look for patterns
empirically. That is what we are about to do — not as a toy exercise, but because it is
genuinely how this problem is attacked.

## The data

21,263 measured superconductors from the NIMS SuperCon database, processed by Hamidieh
(2018). Each is described by 81 features that are *statistical summaries of composition* —
take a property like atomic mass, first ionisation energy, atomic radius, density, electron
affinity, fusion heat, thermal conductivity or valence, and compute its mean, weighted mean,
geometric mean, entropy, range and standard deviation across the elements in the formula.

We are working with a 90% split of it. The remaining 10% is the blind test set for your
final project — you will not see those materials, or their critical temperatures, at any
point this term.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

sc   = pd.read_csv('../../data/superconductor_train.csv.gz')   # 81 features + critical_temp
comp = pd.read_csv('../../data/composition_train.csv.gz')      # per-element amounts + formula

print(sc.shape, comp.shape)
sc.head()


## What does $T_c$ look like?

Start with the target before touching a model.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(sc.critical_temp, bins=60)
ax[0].set_xlabel('$T_c$ (K)'); ax[0].set_ylabel('count'); ax[0].set_title('linear')
ax[1].hist(sc.critical_temp, bins=60)
ax[1].set_yscale('log')
ax[1].set_xlabel('$T_c$ (K)'); ax[1].set_title('log counts')
fig.tight_layout()

print(sc.critical_temp.describe().round(2))


Strongly skewed: most materials have low $T_c$, with a long tail to high values. The mean
sits well above the median.

**What is the single highest-$T_c$ material here?**


In [ ]:
comp.nlargest(8, 'critical_temp')[['material', 'critical_temp']]


H₂S at 185 K — and here is a trap worth naming now.

H₂S superconducts at that temperature only under a pressure of order **a million
atmospheres**. It is a hydride, stabilised by extreme compression. The rows below it —
Hg/Tl/Bi cuprates around 135 K — are ambient-pressure materials.

**There is no pressure column.** The 81 features describe composition and nothing else. So
the most extreme point in this dataset was measured under conditions the features cannot
represent, sitting in the same table as everything else. A model will cheerfully learn
"hydrogen and sulfur means very high $T_c$", which is false at one atmosphere.

This is not a flaw we can fix; it is a limitation to know about. Keep it in mind whenever a
model tells you something confident about hydrogen compounds.


## A physically motivated boundary

We could pick any threshold to classify on, but one is handed to us: the **~40 K BCS
ceiling**. Above it, phonon-mediated pairing is generally not sufficient and a different
mechanism is implicated. So "is this material above 40 K?" is not an arbitrary question —
it is close to asking "is this material *conventional*?"


In [ ]:
y = (sc.critical_temp > 40).astype(int).values
print(f'above 40 K: {y.mean():.1%}  ({y.sum()} of {len(y)})')


## Do the chemical families show up?

`composition_train.csv.gz` has one column per element, so we can sort materials into families
by what they contain — without the model, and without any fitting.


In [ ]:
cuprate = (comp.Cu > 0) & (comp.O > 0)
ironbased = (comp.Fe > 0) & ((comp.As > 0) | (comp.Se > 0) | (comp.P > 0) | (comp.S > 0)) & ~cuprate
hydride = (comp.H > 0) & ~cuprate & ~ironbased
other = ~cuprate & ~ironbased & ~hydride

for name, mask in [('cuprate', cuprate), ('iron-based', ironbased),
                   ('H-containing', hydride), ('everything else', other)]:
    t = comp.critical_temp[mask]
    print(f'{name:16} n={mask.sum():6d}   mean Tc={t.mean():6.1f} K   median={t.median():6.1f} K')

high = comp.critical_temp > 40
print(f'\nof the {high.sum()} materials above 40 K, {100 * (high & cuprate).sum() / high.sum():.0f}% are cuprates')


That last number is the point. The 40 K boundary we chose on theoretical grounds turns out
to separate the cuprates from nearly everything else *empirically*. The physics and the data
agree, and neither was told about the other.

Hold on to this — in the final project you will use exactly this kind of independent check to
judge whether a model's predictions are physically plausible.


## Conditioning: the same model, very different effort

Four orders of magnitude separate the smallest and largest feature. That makes the cost
surface a long narrow valley: a step size sensible in one direction is far too large or far
too small in another.

Whether that *matters* depends on your optimizer — which is a more interesting result than it
sounds. We will measure it rather than assert it.


In [ ]:
X = sc.drop(columns=['id', 'critical_temp']).values.astype(float)
means = np.abs(X.mean(axis=0))
print(f'feature means span {means.min():.2f} to {means.max():.0f}')
print('\nthe five largest:')
for i in np.argsort(means)[-5:][::-1]:
    print(f'  {sc.drop(columns=["id","critical_temp"]).columns[i]:34} {X[:, i].mean():12.1f}')


Four orders of magnitude between the smallest and largest feature. Gradient-based
optimisation on raw features like this is very slow to converge — the cost surface is a long
narrow valley, and a step size that works for one direction is hopeless for another.

We will fit it **both ways** so you can see how much it costs.


## Logistic regression, as in the last notebook

Same cost function and gradient as before, just with 81 features instead of two.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def cost(w, Xb, y):
    p = sigmoid(Xb @ w)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def gradient(w, Xb, y):
    return Xb.T @ (sigmoid(Xb @ w) - y) / len(y)

def fit(Xtrain, ytrain):
    global fit_result                      # so we can inspect iteration counts below
    Xb = np.c_[np.ones(len(Xtrain)), Xtrain]
    w0 = np.zeros(Xb.shape[1])
    fit_result = minimize(cost, w0, args=(Xb, ytrain), jac=gradient, method='L-BFGS-B')
    return fit_result.x

def accuracy(w, Xv, yv):
    Xb = np.c_[np.ones(len(Xv)), Xv]
    return ((sigmoid(Xb @ w) > 0.5) == yv).mean()


### Hold out a validation set

We need somewhere honest to measure performance. **Never judge a model on the data you fitted
it to** — it has already seen those answers, so the score tells you about memorisation rather
than about the model.

Split the training data: fit on one part, measure on the other.


In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
cut = int(0.8 * len(X))
train_i, val_i = idx[:cut], idx[cut:]

baseline = max(y[val_i].mean(), 1 - y[val_i].mean())
print(f'majority-class baseline: {baseline:.3f}   <- anything worse than this is useless')


In [ ]:
w_raw = fit(X[train_i], y[train_i])
print(f'unscaled       accuracy {accuracy(w_raw, X[val_i], y[val_i]):.3f}   '
      f'iterations {fit_result.nit}')

mu, sd = X[train_i].mean(axis=0), X[train_i].std(axis=0)
sd[sd == 0] = 1
Xs = (X - mu) / sd                      # note: scaling learned from TRAIN only

w_scaled = fit(Xs[train_i], y[train_i])
print(f'standardised   accuracy {accuracy(w_scaled, Xs[val_i], y[val_i]):.3f}   '
      f'iterations {fit_result.nit}')


Both reach essentially the same accuracy — **L-BFGS-B is a quasi-Newton method, and it
builds up curvature information that lets it cope with a badly conditioned problem.** What
scaling bought us was *effort*: roughly 1,400 iterations instead of 11,000.

Now try the optimizer you would write yourself. Plain gradient descent has no curvature
information — it only knows the slope, and one step size has to serve all 81 directions.


In [ ]:
def fit_gradient_descent(Xtrain, ytrain, iters=400, lr=0.5):
    Xb = np.c_[np.ones(len(Xtrain)), Xtrain]
    w = np.zeros(Xb.shape[1])
    for _ in range(iters):
        w -= lr * gradient(w, Xb, ytrain)
    return w

w_gd_raw    = fit_gradient_descent(X[train_i],  y[train_i])
w_gd_scaled = fit_gradient_descent(Xs[train_i], y[train_i])
print(f'gradient descent, unscaled     {accuracy(w_gd_raw,    X[val_i],  y[val_i]):.3f}')
print(f'gradient descent, standardised {accuracy(w_gd_scaled, Xs[val_i], y[val_i]):.3f}')
print(f'baseline                       {baseline:.3f}')


**That** is the cost of ignoring conditioning. With 400 steps of gradient descent, the
unscaled fit barely clears the majority-class baseline while the standardised one is close to
what L-BFGS-B achieved.

The lesson is not "always standardise or your model breaks" — a good optimizer rescued us
here. It is that **preconditioning and optimisation are entangled**, and a result that looks
like a modelling failure can just be an optimiser that has not converged. When something
underperforms, check whether it has actually finished before concluding the model is wrong.

Note also *how* the scaling was done: `mu` and `sd` come from the training rows only. Compute
them over all your data and information from the validation set leaks into the fit, making
your score optimistic.


## Accuracy is not enough

About a third of the materials are above 40 K, so the classes are unbalanced. A model that
predicted "below 40 K" for everything would already score around 0.66. Look at *which* errors
are being made.


In [ ]:
Xb_val = np.c_[np.ones(len(val_i)), Xs[val_i]]
pred = (sigmoid(Xb_val @ w_scaled) > 0.5).astype(int)
true = y[val_i]

tp = int(((pred == 1) & (true == 1)).sum()); fp = int(((pred == 1) & (true == 0)).sum())
fn = int(((pred == 0) & (true == 1)).sum()); tn = int(((pred == 0) & (true == 0)).sum())

print(f'                 predicted cold   predicted warm')
print(f'actually cold  {tn:14d} {fp:16d}')
print(f'actually warm  {fn:14d} {tp:16d}')
print(f'\nprecision (of those called warm, how many are): {tp / (tp + fp):.3f}')
print(f'recall    (of the warm ones, how many we found): {tp / (tp + fn):.3f}')


Precision and recall answer different questions, and which one you care about depends on
what the model is *for*. Screening candidate materials for an expensive synthesis? False
positives waste money. Trying not to overlook a promising compound? False negatives are worse.

There is no single "accuracy" that captures this.


## What did it learn?

The coefficients are directly comparable now that the features are standardised — each is
"how much does the log-odds move per standard deviation of this feature".


In [ ]:
names = sc.drop(columns=['id', 'critical_temp']).columns
order = np.argsort(np.abs(w_scaled[1:]))[::-1][:12]
for i in order:
    print(f'  {w_scaled[1 + i]:+7.3f}   {names[i]}')


Do these make physical sense? Which elemental properties would you *expect* to matter for
pairing, and does the model agree? Where it disagrees with your intuition, is that the model
finding something — or an artifact of 81 highly correlated features?

Bear in mind that many of these features are different summaries of the same underlying
property, so a large coefficient on one and a large opposite coefficient on another may just
be the fit playing them off against each other.


---
## Where this goes

The **final project** uses this dataset. You will:

- predict $T_c$ as a continuous quantity, not just a binary label
- classify about the 40 K boundary, and compare that with thresholding your regression
- check your predictions against the chemical families we just looked at
- and submit predictions for **2,126 materials whose $T_c$ you never see**

Everything in this notebook is a starting point for it.
